# MarginGraph: fine-tune Qwen as a reranker (E6) on a free Kaggle GPU

Zero-shot Qwen 1.5B and 3B both lost to LightGBM. This notebook teaches Qwen2.5-1.5B this shop's buying patterns with a LoRA fine-tune, then runs the same go/no-go.

- **Training data:** baskets from the last 90 days of the *train* period. Each basket has one hidden item (Yes) and LightGBM's top wrong guesses (No). Test baskets are never seen.
- **Test:** the same 1,000 test baskets and top-20 candidates as the zero-shot run, paired against LightGBM.

Kaggle settings: **Accelerator: GPU T4 x2**, **Internet: on**, and your Online Retail II dataset added as an input. Then **Run All** (about 45-75 min).

In [ ]:
# 1) Check the GPU, then install (about 2 min). Needs Internet ON in the Kaggle settings panel.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
os.chdir('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
!rm -rf MarginGraph && git clone -q -b claude/project-thread-mkbaip https://github.com/Sanskar1724/MarginGraph.git
%cd MarginGraph
!pip install -q -e . transformers accelerate
# make the package importable in this already-running kernel (editable installs need a restart otherwise)
import sys
sys.path.insert(0, os.path.abspath('src'))
import margingraph; print('margingraph', margingraph.__version__)

In [ ]:
# 2) Find the dataset.
#    Kaggle: add your uploaded dataset (Add Input -> Your Datasets); any online_retail*.xlsx or .csv under /kaggle/input is found.
#    Colab: a file picker opens instead.
import glob, os, shutil
os.makedirs('data/raw', exist_ok=True)
found = sorted(glob.glob('/kaggle/input/**/online_retail*.xlsx', recursive=True)) + \
        sorted(glob.glob('/kaggle/input/**/online_retail*.csv', recursive=True))
if found:
    src = found[0]
elif os.path.isdir('/kaggle/input'):
    raise FileNotFoundError('No online_retail*.xlsx/.csv under /kaggle/input. Click "Add Input" and add your dataset.')
else:
    from google.colab import files
    src = next(iter(files.upload()))
dst = 'data/raw/online_retail_II' + os.path.splitext(src)[1]
shutil.copy(src, dst)
!sed -i "s#raw_path: .*#raw_path: {dst}#" configs/default.yaml
print('using', src, os.path.getsize(dst) // 1_000_000, 'MB')

In [ ]:
# 3) CPU pipeline (GPU sits idle here, that's expected): clean, baselines, LightGBM, Stage 2 (about 10-15 min on Colab)
!OPENBLAS_NUM_THREADS=1 PYTHONPATH=src python -m margingraph.cli run 2>&1 | grep -v Warning
import pandas as pd
pd.read_csv('results/e5_rerank.csv').round(4)

In [ ]:
# 4) Fine-tune (about 20-40 min on one T4)
import pandas as pd, torch
from margingraph import llm as L
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
EPOCHS = 2
catalog = pd.read_parquet('artifacts/catalog.parquet')
train = pd.read_parquet('artifacts/train_slates.parquet')
ex = L.sft_examples(train, catalog, neg_per_pos=3)
print(len(ex), 'training pairs,', int(ex.label.sum()), 'positives')
model, tok = L.load_model(MODEL)
model = L.finetune_lora(model, tok, ex, epochs=EPOCHS, lr=2e-4, batch_size=8)
model.save_pretrained('qwen1.5b_marginGraph_lora')   # small adapter, in the Output panel

In [ ]:
# 5) Go/no-go on the same 1,000 test baskets as the zero-shot run
slates = pd.read_parquet('artifacts/test_slates.parquet')
ids = slates.event_id.drop_duplicates().sample(1000, random_state=0)
sub = slates[slates.event_id.isin(ids)].copy()
sub['llm'] = L.score_slates(sub, catalog, model=model, tok=tok)
table = L.compare(sub)
display(table.round(4)); print(L.verdict(table))

In [ ]:
# 6) Does the fine-tuned LLM add signal on top of LightGBM?
val = pd.read_parquet('artifacts/val_slates.parquet')
val['llm'] = L.score_slates(val, catalog, model=model, tok=tok)
st, coef = L.stack(val, sub)
display(st.round(4)); display(coef.round(3)); print(L.verdict(st))
sub.to_parquet('llm_ft_test_scores.parquet'); val.to_parquet('llm_ft_val_scores.parquet')

## 7) Send the result back

Paste the last training-loss line, both tables, and both GO/NO-GO lines into the project thread.